# 4. 로그인 상태 저장하기

로그인한 사용자를 매번 인증 서버에 물어보면 그때마다 시간이 걸린다.
한 번 확인한 결과를 Redis 에 담아두고, 일정 시간이 지나면 다시 확인한다.

다음 차수에서 FastAPI 서버에 그대로 적용한다.

위에서부터 셀을 하나씩 실행합니다 (`Shift + Enter`).

TODO 를 채우기 전에는 결과가 비어 있게 나온다 (None, [], {}).
`...` 은 파이썬 문법상 유효해서 오류 없이 지나가기 때문이다.
결과가 비어 있으면 고장난 것이 아니라 아직 안 채운 것이다.

In [ ]:
import hashlib
import json
import time

from redis_client import r



def make_key(token):
    """토큰으로 키를 만든다. 토큰을 그대로 쓰지 않고 해시로 바꾼다."""
    # TODO 1. 토큰을 sha256 으로 바꿔 "day14:session:" 뒤에 붙인다.
    #         힌트: sha256 은 문자열이 아니라 bytes 를 받는다 (token.encode()).
    #               결과는 hexdigest() 로 문자열이 된다.
    #
    #         아래 return 을 지우고 직접 작성한다.
    #         지금은 해시 없이 토큰을 그대로 쓰고 있다. 5절에서 왜 안 되는지 확인한다.
    return "day14:session:" + token


def ask_auth_server(token):
    """인증 서버에 물어보는 것을 흉내낸다. 일부러 느리게 만들었다."""
    time.sleep(0.3)

    if not token.startswith("valid-"):
        return None

    user_id = token.replace("valid-", "")
    return {"id": user_id, "email": user_id + "@example.com"}


def get_user(token):
    """로그인한 사용자를 가져온다. 캐시에 있으면 캐시에서."""
    key = make_key(token)

    # TODO 2. 캐시에 있으면 "캐시에서 가져옴" 을 출력하고 그 값을 돌려준다.

    # TODO 3. 없으면 "인증 서버에 물어봄" 을 출력하고 ask_auth_server(token) 을 부른다.
    #         결과가 None 이면 "토큰이 잘못됐다" 를 출력하고 None 을 돌려준다.
    #         캐시에 넣지 않는다.
    #         정상이면 5초 만료로 캐시에 넣고 돌려준다.

    # 아래는 TODO 를 채우기 전에도 끝까지 돌게 하는 임시 코드다. 지우고 작성한다.
    return None


token = "valid-user123"

## 1. 같은 사람이 여러 번 요청하면

In [ ]:
r.delete(make_key(token))

print("1번째 요청")
start = time.time()
user = get_user(token)
print("  걸린 시간:", round(time.time() - start, 3), "초")

print("2번째 요청")
start = time.time()
get_user(token)
print("  걸린 시간:", round(time.time() - start, 3), "초")

print("3번째 요청")
start = time.time()
get_user(token)
print("  걸린 시간:", round(time.time() - start, 3), "초")

print()
print("첫 요청만 인증 서버까지 갔다. 나머지는 Redis 에서 바로 답했다.")

## 2. 5초가 지나면

In [ ]:
print("5초 기다린다...")
time.sleep(6)

print("다시 요청")
get_user(token)
print()
print("캐시가 사라져서 다시 인증 서버로 갔다.")

## 3. 잘못된 토큰

In [ ]:
get_user("이건-가짜-토큰")

print()
print("실패한 결과는 캐시에 넣지 않는다.")
print("넣어두면 나중에 토큰이 정상이 되어도 계속 막힌다.")

## 4. 로그아웃

In [ ]:
get_user(token)
print("로그인 상태의 캐시:", r.exists(make_key(token)))

# TODO 4. 로그아웃 — 이 토큰의 캐시를 지운다.  힌트: r.delete(make_key(token))
...

print("로그아웃 후 캐시:", r.exists(make_key(token)))

print()
print("다시 요청하면")
get_user(token)

print()
print("TTL 을 기다리지 않고 바로 지우는 것이 로그아웃이다.")

## 5. 토큰을 키에 그대로 쓰면 안 되는 이유

In [ ]:
print("토큰 원문:", token)
print("실제 키  :", make_key(token))
print()
print("토큰은 그 자체가 로그인 자격이다.")
print("키 목록에 원문이 남으면 Redis 를 볼 수 있는 사람이 남의 계정을 쓸 수 있다.")
print("해시로 바꾸면 원문이 보이지 않고, 같은 토큰은 항상 같은 해시가 된다.")

## 6. 정리

In [ ]:
keys = r.keys("day14:*")
for key in keys:
    r.delete(key)

print(len(keys), "개 삭제")
print("남은 키 개수:", r.dbsize())